# 04 — Classing Woe

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The bins. Each candidate attribute is coarse-classed on the training vintages with the registry's direction as the monotone constraint, minimum population and bad counts per bin, and a missing bin where it earns one. WOE is the log-likelihood ratio between goods and bads per bin, IV its Jeffreys divergence; both are explained in the Claude Doc section 5.

Decisions: **F — binning parameters**, **G — overrides of specific bin edges**, **H — the IV cut and the stability (CSI) screen**. The output is `models/bins.json`.

In [ ]:
frame = load_frame()
tr = frame[frame.split == "train"]
cands = json.loads((MODELS / "candidate_attributes.json").read_text(encoding="utf-8"))["candidates"]
print(len(cands), "candidates;", "train", tr.shape)

### Decision F — Binning parameters

`max_bins` (non-missing bins), `min_bin_share` (share of train per bin), `min_bads` (bads per bin, the moment-polytope guard). Siddiqi's rule of thumb is at most 10 bins and at least 5% per bin; the bad-count floor matters more than the share here because the bad rate is ~10%.

**Your reasoning:**

_(write here)_

In [ ]:
PARAMS = dict(max_bins=8, prebins=20, min_bin_share=0.05, min_bads=50)
specs = B.fit_all(tr, cands, directions=DIRECTIONS, **PARAMS)
ivs = B.iv_summary(specs); display(ivs)

In [ ]:
# Evidence — WOE by bin for the strongest attributes (tables and a plot each)
top = ivs.attribute.head(12).tolist()
for n in top:
    t = specs[n].table(); print(f"\n{n}  IV={specs[n].iv:.4f}  direction={specs[n].direction}  monotone={specs[n].monotone}  ({MEANINGS.get(n, '')})")
    display(t)
fig, axes = plt.subplots(3, 4, figsize=(16, 9)); axes = axes.ravel()
for ax, n in zip(axes, top):
    t = specs[n].table(); ax.bar(range(len(t)), t.woe); ax.set_xticks(range(len(t))); ax.set_xticklabels(t.label, rotation=60, ha="right", fontsize=7); ax.set_title(f"{n} (IV {specs[n].iv:.3f})", fontsize=9)
plt.tight_layout(); plt.show()

### Decision G — Overrides

Where a bin edge should sit on a business boundary (utilisation at 30/50/75/90, inquiries at 0/1/2/3+), or where the automatic merge produced a bin you cannot explain, set the edges by hand. Non-monotone shapes stay only with a documented reason.

**Your reasoning:**

_(write here)_

In [ ]:
OVERRIDES = {
    # "revol_util": [30, 50, 75, 90],
    # "inq_last_6mths": [0, 1, 2],
}
for n, edges in OVERRIDES.items():
    specs[n] = B.fit_numeric_with_edges(tr[n], tr["bad"], edges, name=n, min_bads=PARAMS["min_bads"])
    print(n); display(specs[n].table())
ivs = B.iv_summary(specs)

In [ ]:
# Evidence — stability of the bins across vintages (CSI of each attribute's bin shares vs train)
csi_tbl = B.csi_by_group(frame, specs, group_col="vintage_year")
display(csi_tbl.style.background_gradient(cmap="Reds", axis=None) if hasattr(csi_tbl, "style") else csi_tbl)
unstable = csi_tbl[(csi_tbl > 0.25).any(axis=1)].index.tolist()
print("attributes with CSI > 0.25 in some vintage:", unstable)

### Decision H — IV cut and stability screen

Keep attributes above which IV? Drop those whose bins are unstable across vintages (a missing bin that disappears in 2016 is a structural gap, not consumer behaviour)? Attributes with IV above 0.5 are suspicious and need a leakage explanation.

**Your reasoning:**

_(write here)_

In [ ]:
MIN_IV = 0.02
DROP = list(unstable)            # edit: keep an unstable attribute only with a reason written above
final = [n for n in ivs[ivs.iv >= MIN_IV].attribute if n not in DROP]
B.save_specs({n: specs[n] for n in final}, MODELS / "bins.json")
(MODELS / "binning_decisions.json").write_text(json.dumps({"binned_attributes": final, "params": PARAMS, "overrides": OVERRIDES, "min_iv": MIN_IV, "dropped_unstable": DROP}, indent=1), encoding="utf-8")
ivs[ivs.attribute.isin(final)].to_csv(config.DOCS_DIR / "exhibits" / "iv_table.csv", index=False)
print(len(final), "attributes binned and saved to", MODELS / "bins.json")